# TensorFlow/Keras class-imbalance utilities

Reusable helpers for training with imbalanced image classes: class-weight calculation, training-only oversampling, and training-only augmentation. The code below is organized into executable cells and mirrors `imbalance.py`.

Input datasets should yield `(images, sparse_integer_labels)`. The class IDs used by the project are `NORMAL=0` and `PNEUMONIA=1`. Keep validation and test datasets unbalanced and unaugmented.

In [ ]:
"""TensorFlow/Keras utilities for training with imbalanced image classes."""

from __future__ import annotations

from collections.abc import Sequence

import tensorflow as tf




## Count training labels

Private helper: validates a finite training dataset and counts sparse class IDs.

In [ ]:
def _count_training_labels(
    dataset: tf.data.Dataset,
    num_classes: int,
) -> list[int]:
    """Count sparse integer class IDs in a finite batched dataset."""
    if num_classes < 2:
        raise ValueError("num_classes must be at least 2.")

    cardinality = int(tf.data.experimental.cardinality(dataset).numpy())
    if cardinality == int(tf.data.experimental.INFINITE_CARDINALITY):
        raise ValueError("The training dataset must be finite, not repeated.")

    counts = tf.zeros([num_classes], dtype=tf.int64)
    total = 0
    for batch in dataset:
        if not isinstance(batch, (tuple, list)) or len(batch) != 2:
            raise ValueError("Expected a dataset of (images, sparse_integer_labels).")

        labels = tf.convert_to_tensor(batch[1])
        if not labels.dtype.is_integer:
            raise ValueError("Labels must be sparse integer class IDs.")
        if labels.shape.rank is not None and labels.shape.rank > 1:
            if labels.shape[-1] != 1:
                raise ValueError("One-hot labels are unsupported; use sparse integer labels.")

        labels = tf.cast(tf.reshape(labels, [-1]), tf.int64)
        if int(tf.size(labels).numpy()) == 0:
            continue
        if bool(
            tf.reduce_any(
                tf.logical_or(labels < 0, labels >= num_classes)
            ).numpy()
        ):
            raise ValueError(f"Labels must be in the range [0, {num_classes - 1}].")

        counts += tf.math.bincount(
            labels,
            minlength=num_classes,
            maxlength=num_classes,
            dtype=tf.int64,
        )
        total += int(tf.size(labels).numpy())

    if total == 0:
        raise ValueError("The training dataset contains no labels.")
    return [int(count) for count in counts.numpy()]




## Class-weighted loss

Compute inverse-frequency weights for `model.fit(class_weight=...)` from training labels only.

In [ ]:
def compute_class_weights(
    train_dataset: tf.data.Dataset,
    num_classes: int = 2,
) -> dict[int, float]:
    """Return inverse-frequency weights for ``model.fit(class_weight=...)``.

    Counts are collected only from ``train_dataset``. Every class ID from
    ``0`` through ``num_classes - 1`` must be present; missing classes raise a
    clear error rather than producing misleading weights.
    """
    counts = _count_training_labels(train_dataset, num_classes)
    missing = [class_id for class_id, count in enumerate(counts) if count == 0]
    if missing:
        raise ValueError(
            "Cannot compute class weights because training classes are absent: "
            f"{missing}. Check the training split and num_classes."
        )

    total = sum(counts)
    return {
        class_id: total / (num_classes * count)
        for class_id, count in enumerate(counts)
    }




## Training-only oversampling

Repeat minority-class examples in memory to return a balanced training dataset.

In [ ]:
def oversample_training_dataset(
    train_dataset: tf.data.Dataset,
    *,
    num_classes: int = 2,
    target_per_class: int | None = None,
    batch_size: int = 32,
    seed: int | None = 42,
    shuffle_buffer_size: int = 1000,
) -> tf.data.Dataset:
    """Repeat training examples to create an exactly balanced finite dataset.

    The input must be a finite, batched ``(images, sparse_labels)`` dataset.
    No files are copied. The returned dataset has ``target_per_class`` examples
    per class and is batched at ``batch_size``.
    """
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    if shuffle_buffer_size < 1:
        raise ValueError("shuffle_buffer_size must be positive.")

    counts = _count_training_labels(train_dataset, num_classes)
    missing = [class_id for class_id, count in enumerate(counts) if count == 0]
    if missing:
        raise ValueError(
            "Cannot oversample because training classes are absent: "
            f"{missing}. Check the training split and num_classes."
        )

    max_count = max(counts)
    if target_per_class is None:
        target_per_class = max_count
    if target_per_class < max_count:
        raise ValueError(
            "target_per_class must be at least the largest original class count "
            f"({max_count}); this function does not undersample."
        )

    unbatched = train_dataset.unbatch()
    class_datasets = []
    for class_id in range(num_classes):
        class_dataset = unbatched.filter(
            lambda _image, label, expected=class_id: tf.equal(
                tf.cast(tf.reshape(label, []), tf.int64), expected
            )
        )
        class_datasets.append(class_dataset.repeat().take(target_per_class))

    balanced = class_datasets[0]
    for class_dataset in class_datasets[1:]:
        balanced = balanced.concatenate(class_dataset)
    return balanced.shuffle(
        buffer_size=min(num_classes * target_per_class, shuffle_buffer_size),
        seed=seed,
        reshuffle_each_iteration=True,
    ).batch(batch_size)




## Training-only augmentation

Apply conservative, shape-preserving random image transforms while retaining labels.

In [ ]:
def augment_training_dataset(
    train_dataset: tf.data.Dataset,
    *,
    rotation_factor: float = 0.02,
    zoom_factor: float = 0.05,
    translation_factor: float = 0.02,
    seed: int | None = None,
) -> tf.data.Dataset:
    """Apply conservative, shape-preserving random transforms to training images.

    Rotation is a fraction of a full turn; zoom and translation are fractions of
    image dimensions. Labels are returned unchanged. Apply this function only to
    the training dataset; validation and test datasets are not modified.
    """
    factors = {
        "rotation_factor": (rotation_factor, 0.5),
        "zoom_factor": (zoom_factor, 1.0),
        "translation_factor": (translation_factor, 1.0),
    }
    for name, (value, maximum) in factors.items():
        if not 0 <= value <= maximum:
            raise ValueError(f"{name} must be between 0 and {maximum}.")

    seeds: Sequence[int | None]
    if seed is None:
        seeds = (None, None, None)
    else:
        seeds = (seed, seed + 1, seed + 2)

    layers = []
    if rotation_factor:
        layers.append(
            tf.keras.layers.RandomRotation(
                rotation_factor,
                fill_mode="reflect",
                seed=seeds[0],
            )
        )
    if zoom_factor:
        layers.append(
            tf.keras.layers.RandomZoom(
                height_factor=(-zoom_factor, zoom_factor),
                width_factor=(-zoom_factor, zoom_factor),
                fill_mode="reflect",
                seed=seeds[1],
            )
        )
    if translation_factor:
        layers.append(
            tf.keras.layers.RandomTranslation(
                height_factor=translation_factor,
                width_factor=translation_factor,
                fill_mode="reflect",
                seed=seeds[2],
            )
        )

    if not layers:
        return train_dataset

    augmenter = tf.keras.Sequential(layers)

    def augment_batch(images: tf.Tensor, labels: tf.Tensor):
        augmented = augmenter(images, training=True)
        augmented.set_shape(images.shape)
        return augmented, labels

    return train_dataset.map(
        augment_batch,
        num_parallel_calls=tf.data.AUTOTUNE,
    )
